In [0]:
%sh
nc -zv c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com 443

Connection to c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com (52.191.218.117) 443 port [tcp/https] succeeded!


In [0]:
%sh curl -s ifconfig.me

52.249.199.78

In [0]:
from sdds.common.util import NotebookUtil
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType
from databricks.sdk.runtime import dbutils

import json
import requests
from requests.auth import HTTPBasicAuth
from datetime import datetime, timedelta

catalog_name = NotebookUtil.notebook_param("sdds_catalog")
schema_name = NotebookUtil.notebook_param("sdds_bronze_schema")
table_name = "catalog-load-dbx-bronze"

# ES connection config for QA cluster: dsg-search-qa-east
es_host = "d89a8095f4ec40d8ac0443696bbcb049.eastus.azure.elastic-cloud.com"
es_port = "443"
es_user = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="sdsc-search-es-user-prodauth")
es_password_key = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="sdsc-search-es-password-prodauth")
es_index = "catalog-load-read"

In [0]:
es_fields = [
    "parentPartnumber",
    "partnumber",
    "parentCatentryId",
    "catentryId",
    "brand",
    "color",
    "colorFamily",
    "colorSwatch",
    "colorSeq",
    "dsgId",
    "dsgIdentifier",
    "type",
    "attributes",
    "customSkuAttributes",
    "defAttributes",
    "floatFacets",
    "numberFacets",
    "searchAttributes",
    "stringFacets",
    "assetSeoUrl",
    "catgroupSeq",
    "dsgCatgroups",
    "dsgSeoUrl",
    "ggCatgroups",
    "ggSeoUrl",
    "leafCategories",
    "parentCatgroup",
    "parentCatgroup0",
    "parentCatgroup1",
    "parentCatgroup2",
    "parentCatgroup3",
    "parentCatgroup4",
    "parentCatgroup5",
    "parentCatgroup6",
    "parentCatgroup7",
    "parentCatgroup8",
    "parentCatgroup9",
    "plCatgroups",
    "plSeoUrl",
    "primaryCategories",
    "productGroup",
    "productSearchFlag",
    "seo",
    "seoURLs",
    "buyable",
    "catalogIds",
    "dsgProductSortDate",
    "dsgPublishOverride",
    "endDate",
    "endDateTime",
    "fullImage",
    "ggProductSortDate",
    "ggPublishOverride",
    "keyword",
    "longDescription",
    "mfName",
    "name",
    "onOrder",
    "comingSoonEndDateTime",
    "plProductSortDate",
    "plPublishOverride",
    "productType",
    "published",
    "startDate",
    "startDateTime",
    "taxCode",
    "thumbnail",
    "webActiveDate",
    "ggOverrides",
    "plOverrides",
    "dsgPriceIndicators",
    "ggPriceIndicators",
    "plPriceIndicators",
    "kafkaPriceList",
    "priceList",
    "dsgQuantitySold",
    "dsgTotalPriceSold",
    "dsgOverrides",
    "ggQuantitySold",
    "ggTotalPriceSold",
    "plQuantitySold",
    "plTotalPriceSold",
    "salesData",
    "ranking",
    "caliaWebActive",
    "dsgAppWebActive",
    "dsgMobileAppWebActive",
    "dsgWebActive",
    "g3WebActive",
    "ggAppWebActive",
    "ggMobileAppWebActive",
    "ggWebActive",
    "ggAkamaiRedirect",
    "ggKeywordOverride",
    "ggUrl",
    "plWebActive",
    "stackdWebActive",
    "vrstWebActive",
    "swatchPartNumber",
    "primaryUPC",
    "auxDescription2",
]

# Schema: all fields as StringType for bronze-layer raw ingestion
schema = StructType([StructField(f, StringType(), True) for f in es_fields])

# --- Step 1: Scroll ES data and write batches to temp DBFS path ---
base_url = f"https://{es_host}:{es_port}"
auth = HTTPBasicAuth(es_user, es_password_key)
headers = {"Content-Type": "application/json"}

load_ts = datetime.now()
load_timestamp = load_ts.strftime("%Y-%m-%d %H:%M:%S")

# Volume-based landing path partitioned by extraction date (from parameter widget)
volume_base = f"/Volumes/{catalog_name}/{schema_name}/{table_name}"
dbutils.widgets.text("extraction_date", datetime.now().strftime('%Y-%m-%d'))
extraction_date = dbutils.widgets.get("extraction_date")
volume_path = f"{volume_base}/extraction_date={extraction_date}"
dbutils.fs.mkdirs(volume_path)

search_body = {
    "size": 5000,
        "query": {"match_all": {}},
    "_source": es_fields
}

response = requests.post(f"{base_url}/{es_index}/_search?scroll=5m", json=search_body, auth=auth, headers=headers)
if not response.ok:
    raise RuntimeError(f"Elasticsearch search failed: {response.status_code} {response.text}")
results = response.json()
scroll_id = results["_scroll_id"]
hits = results["hits"]["hits"]
total_hits = results["hits"]["total"]["value"]
print(f"Total matching documents: {total_hits}")

batch_num = 0
total_fetched = 0

def write_batch(hits_batch, batch_id):
    """Write a batch of hits as newline-delimited JSON to DBFS."""
    records = []
    for hit in hits_batch:
        src = hit["_source"]
        # Convert all values to strings for consistent bronze-layer ingestion
        record = {k: json.dumps(v) if isinstance(v, (list, dict)) else str(v) if v is not None else None for k, v in src.items()}
        records.append(json.dumps(record))
    lines = "\n".join(records)
    dbutils.fs.put(f"{volume_path}/batch_{batch_id:05d}.json", lines, overwrite=True)
    return len(hits_batch)

# Write initial batch
if hits:
    total_fetched += write_batch(hits, batch_num)
    batch_num += 1

# Scroll through remaining results
while len(hits) > 0:
    response = requests.post(f"{base_url}/_search/scroll", json={"scroll": "5m", "scroll_id": scroll_id}, auth=auth, headers=headers)
    response.raise_for_status()
    results = response.json()
    scroll_id = results.get("_scroll_id")
    hits = results["hits"]["hits"]
    if hits:
        total_fetched += write_batch(hits, batch_num)
        batch_num += 1
        #if batch_num == 5:
        #   break
        if total_fetched % 50000 < 5000:
            print(f"  Fetched {total_fetched} / {total_hits} documents...")

# Clear scroll context
requests.delete(f"{base_url}/_search/scroll", json={"scroll_id": scroll_id}, auth=auth, headers=headers)
print(f"Fetched {total_fetched} documents in {batch_num} batches.")

if total_fetched == 0:
    dbutils.notebook.exit("No new records. Exiting.")


Total matching documents: 3783266
Wrote 155188 bytes.
Wrote 109999 bytes.
Wrote 156136 bytes.
Wrote 131720 bytes.
Wrote 121453 bytes.
Wrote 114536 bytes.
Wrote 166785 bytes.


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# --- Step 2: Read JSON files from volume and add load timestamp ---
volume_base = f"/Volumes/{catalog_name}/{schema_name}/{table_name}"
volume_path = f"{volume_base}/extraction_date={extraction_date}"

df = (
    spark.read.schema(schema).json(volume_path)
    .withColumn("load_timestamp", lit(load_timestamp).cast("timestamp"))
    .withColumn("extraction_date", lit(extraction_date).cast("date"))
    .filter(col("type").isin("style", "sku", "bundle"))
)

record_count = df.count()
print(f"DataFrame record count: {record_count}")
df.display()

In [0]:
# --- Step 3: Write extraction data to Delta table, partitioned by extraction_date (idempotent) ---
target_table = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

table_exists = spark.catalog.tableExists(f"{catalog_name}.{schema_name}.`{table_name}`")

# Migration: drop table if it exists with wrong (or no) partition column
if table_exists:
    partition_cols = spark.sql(f"DESCRIBE DETAIL {target_table}").select("partitionColumns").collect()[0][0]
    if "extraction_date" not in str(partition_cols):
        spark.sql(f"DROP TABLE IF EXISTS {target_table}")
        table_exists = False
        print(f"Dropped {target_table} — wrong partition ({partition_cols}), will recreate with extraction_date")

if not table_exists:
    # First run: create table with extraction_date partition
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("extraction_date")
        .saveAsTable(target_table)
    )
else:
    # Subsequent runs: overwrite only this partition (idempotent re-runs)
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"extraction_date = '{extraction_date}'")
        .partitionBy("extraction_date")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

print(f"Wrote {record_count} records to {target_table} (partition: extraction_date = '{extraction_date}')")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:468)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:571)
	at com.data